# Spesenabrechnungsanalyse

Dieses Notebook zeigt, wie man Agenten erstellt, die Plugins verwenden, um Reisekosten aus lokalen Belegbildern zu verarbeiten, eine Spesenabrechnungs-E-Mail zu generieren und Spesendaten mithilfe eines Tortendiagramms zu visualisieren. Agenten wählen Funktionen dynamisch basierend auf dem Aufgabenkontext aus.

Schritte:
1. OCR-Agent verarbeitet das lokale Belegbild und extrahiert Reisekostendaten.
2. E-Mail-Agent generiert eine Spesenabrechnungs-E-Mail.

### Beispiel eines Reiseszenarios:
Stellen Sie sich vor, Sie sind ein Mitarbeiter, der für ein Geschäftsmeeting in eine andere Stadt reist. Ihr Unternehmen hat eine Richtlinie, alle angemessenen reisekostenbezogenen Ausgaben zu erstatten. Hier ist eine Aufschlüsselung möglicher Reisekosten:
- Transport:
Flugkosten für eine Hin- und Rückfahrt von Ihrer Heimatstadt zur Zielstadt.
Taxi- oder Mitfahrdienste zum und vom Flughafen.
Lokale Verkehrsmittel in der Zielstadt (wie öffentlicher Nahverkehr, Mietwagen oder Taxis).

- Unterkunft:
Hotelaufenthalt für drei Nächte in einem mittelklassigen Business-Hotel in der Nähe des Tagungsortes.

- Mahlzeiten:
Tägliche Mahlzeitenpauschale für Frühstück, Mittag- und Abendessen, basierend auf der Pauschalregelung des Unternehmens.

- Sonstige Ausgaben:
Parkgebühren am Flughafen.
Internetgebühren im Hotel.
Trinkgelder oder kleine Servicegebühren.

- Dokumentation:
Sie reichen alle Belege (Flüge, Taxis, Hotel, Mahlzeiten usw.) sowie einen ausgefüllten Spesenbericht zur Erstattung ein.


## Erforderliche Bibliotheken importieren

Importieren Sie die erforderlichen Bibliotheken und Module für das Notebook.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Definiere Ausgabenmodelle

 Erstelle ein Pydantic-Modell für einzelne Ausgaben und eine ExpenseFormatter-Klasse, um eine Benutzeranfrage in strukturierte Ausgabendaten umzuwandeln.

 Jede Ausgabe wird im Format dargestellt:
 `{'date': '07-Mar-2025', 'description': 'flug zum zielort', 'amount': 675.99, 'category': 'Transport'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Definition von Tools - Erstellen der E-Mail

Erstellen Sie eine Tool-Funktion, um eine E-Mail zur Einreichung einer Spesenabrechnung zu generieren.
- Dieses Tool verwendet den `@tool` Dekorator aus dem Microsoft Agent Framework.
- Es berechnet den Gesamtbetrag der Ausgaben und formatiert die Details in einen E-Mail-Text.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Werkzeug zum Extrahieren von Reisekosten aus Belegbildern

Erstellen Sie eine Werkzeugfunktion zum Extrahieren von Reisekosten aus Belegbildern.
- Dieses Werkzeug verwendet den `@tool` Dekorator des Microsoft Agent Frameworks.
- Es liest das Belegbild, kodiert es als base64 und gibt die Daten-URI zurück, damit der Agent sie analysieren kann.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Verarbeitung von Ausgaben

Definieren Sie die Agenten und verbinden Sie sie zu einem sequentiellen Arbeitsablauf mit `WorkflowBuilder`.
- Der OCR-Agent extrahiert strukturierte Ausgabendaten aus dem Belegbild mit dem Tool `load_receipt_image`.
- Der E-Mail-Agent nimmt die extrahierten Daten und erstellt eine professionelle Spesenabrechnung per E-Mail mit dem Tool `generate_expense_email`.
- `WorkflowBuilder` mit `add_edge` erstellt eine sequentielle Pipeline: OCR-Agent → E-Mail-Agent.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Hauptfunktion

Erstellen Sie den sequentiellen Arbeitsablauf und führen Sie ihn aus, um das Bild des Belegs zu verarbeiten und die Spesenabrechnungs-E-Mail zu erstellen.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:  
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Originalsprache gilt als maßgebliche Quelle. Für kritische Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die durch die Nutzung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
